# EEG-CogState — Sprint 3 : Extraction de features

**Objectif du Sprint 3** (cahier des charges) : transformer chaque epoch en un vecteur de nombres exploitable par le modèle.

Pour chaque epoch et chaque canal, on calcule :
1. **Puissance des 5 bandes** (delta, thêta, alpha, bêta, gamma)
2. **Ratios inter-bandes** (bêta/alpha, thêta/bêta…)
3. **Indicateurs statistiques** (variance, Hjorth, entropie spectrale)

Le résultat est un **tableau** : une ligne par epoch, une colonne par feature. C'est ce tableau qui nourrira le modèle au Sprint 4.

Tout est implémenté dans `src/features.py`.

## 1. Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, io_eeg, preprocessing, features

print("Bandes de frequence :", list(config.FREQ_BANDS.keys()))
print("Canaux :", config.N_CHANNELS)

## 2. Features d'un seul epoch

Pour comprendre, on commence petit : on prend **un** epoch et on regarde
les features extraites pour **un** canal.

In [ ]:
catalogue = io_eeg.list_recordings()
sample_path = catalogue.iloc[0]["path"]

# Pretraitement -> epochs (Sprint 2)
result = preprocessing.preprocess_file(sample_path)
first_epoch = result["epochs"][0]   # premier epoch, forme (256, 14)

print("Fichier :", catalogue.iloc[0]["filename"], "| etat :", result["label"])
print("Forme d'un epoch :", first_epoch.shape, "(echantillons x canaux)\n")

# Features du canal O1 uniquement, pour cet epoch
sig_O1 = first_epoch[:, config.CHANNEL_NAMES.index("O1")]
powers = features.band_powers(sig_O1)
ratios = features.band_ratios(powers)

print("Puissances des bandes (canal O1) :")
for band in config.FREQ_BANDS:
    print(f"   {band:6s} : {powers[band + '_power']:8.2f}  "
          f"(relatif : {powers[band + '_rel']*100:5.1f}%)")
print("\nRatios :")
for k, v in ratios.items():
    print(f"   {k:24s} : {v:.3f}")

## 3. Construction du tableau complet

On applique l'extraction à **tous les epochs de tous les fichiers**.
Chaque ligne du tableau = un epoch ; chaque colonne = une feature.

> Cette cellule peut prendre quelques dizaines de secondes selon le nombre de fichiers.

In [ ]:
print("Extraction en cours...\n")
table = features.build_feature_table(catalogue)

print(f"\nTableau : {table.shape[0]} epochs x {table.shape[1]} colonnes")
print(f"   ({table.shape[1] - 2} features + colonnes 'subject' et 'label')")

In [ ]:
# Apercu : les 2 colonnes meta + quelques colonnes de features
cols_apercu = ["subject", "label", "O1_alpha_power", "O1_beta_power",
               "O1_ratio_beta_alpha", "O1_spectral_entropy"]
table[cols_apercu].head(8)

In [ ]:
# Repartition des classes (doit etre relativement equilibree)
print(table["label"].value_counts())

## 4. Les features séparent-elles les états ?

C'est la question clé : si les features ne distinguent pas les classes,
le modèle ne pourra rien apprendre. On compare la distribution de quelques
features entre relaxation et concentration.

**Attendu :** plus d'alpha en relaxation, plus de bêta en concentration.

In [ ]:
feats_a_tracer = ["O1_alpha_power", "O1_beta_power", "O1_ratio_beta_alpha"]
labels_uniques = table["label"].unique()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, feat in zip(axes, feats_a_tracer):
    data = [table[table["label"] == lab][feat].values for lab in labels_uniques]
    ax.boxplot(data, labels=labels_uniques)
    ax.set_title(feat, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", rotation=15)
fig.suptitle("Distribution des features par etat cognitif", fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# Comparaison chiffree des moyennes par classe
comparaison = table.groupby("label")[feats_a_tracer].mean().round(3)
print("Moyenne de chaque feature par etat :\n")
print(comparaison.to_string())

## 5. Sauvegarde du tableau

On enregistre le tableau de features en CSV dans `data/processed/`.
C'est le fichier d'entrée du **Sprint 4** (machine learning).

In [ ]:
out_path = features.save_feature_table(table)
print("Tableau sauvegarde :", out_path)
print("Taille :", table.shape)

## ✅ Bilan du Sprint 3

À ce stade, on sait :
- calculer la puissance des bandes, les ratios et les indicateurs statistiques d'un signal ;
- construire un **tableau de features** complet (un epoch par ligne) ;
- vérifier que les features **séparent** bien les états cognitifs ;
- sauvegarder ce tableau pour la suite.

Le signal est maintenant transformé en **nombres exploitables**.

**Prochaine étape (Sprint 4)** : entraîner un modèle (Random Forest) sur ce tableau, évalué **par sujet** (Leave-One-Subject-Out), et mesurer ses performances (macro-F1, matrice de confusion).